# Time-Series Trend Visualization Milestone

This notebook demonstrates how to identify and visualize trends over time using line plots.

## Learning Objectives
- Understand what time-series data represents
- Visualize data changes over time using line plots
- Identify upward, downward, or stable trends
- Interpret patterns such as spikes or drops
- Build intuition for temporal analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Understanding Time-Based Data

Time-series data has a temporal component that adds context. The order matters!

In [ ]:
# Generate sample time-series data
dates = pd.date_range(start='2024-01-01', end='2024-12-31', freq='W')
n = len(dates)

# Create trend with seasonal pattern and noise
trend = np.linspace(100, 150, n)
seasonal = 10 * np.sin(np.linspace(0, 4*np.pi, n))
noise = np.random.normal(0, 3, n)
sales = trend + seasonal + noise

# Add anomalies
sales[10] = 180  # Spike
sales[25] = 70   # Drop

df = pd.DataFrame({
    'date': dates,
    'sales': sales,
    'temperature': np.random.normal(20, 5, n)
})

print("Dataset created:")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

## 2. Identifying Temporal Columns

First step: identify which columns contain time/date information.

In [ ]:
# Check data types
print("Column data types:")
print(df.dtypes)

# Identify temporal columns
temporal_cols = []
for col in df.columns:
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        temporal_cols.append(col)
    elif any(keyword in col.lower() for keyword in ['date', 'time', 'year']):
        temporal_cols.append(col)

print(f"\nTemporal columns: {temporal_cols}")

## 3. Ensuring Correct Temporal Ordering

**Critical:** Always sort data by time before analysis!

In [ ]:
# Sort by date
df_sorted = df.sort_values('date').reset_index(drop=True)

print("Data sorted by date:")
print(df_sorted.head())
print(f"\nAll {len(df)} data points preserved after sorting")

## 4. Creating Line Plots

Line plots emphasize continuity and show how values change over time.

In [ ]:
# Create basic line plot
plt.figure(figsize=(12, 6))
plt.plot(df_sorted['date'], df_sorted['sales'], 
         linewidth=2, marker='o', markersize=4, color='#2E86AB')

plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Sales', fontsize=12, fontweight='bold')
plt.title('Sales Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("✓ Line plot shows continuous trend over time")

## 5. Identifying Trends

Look for the overall direction: upward, downward, or stable.

In [ ]:
# Calculate trend using linear regression
x = np.arange(len(df_sorted))
y = df_sorted['sales'].values
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
trend_line = p(x)

# Calculate statistics
slope = z[0]
start_val = df_sorted['sales'].iloc[0]
end_val = df_sorted['sales'].iloc[-1]
percent_change = ((end_val - start_val) / start_val) * 100

# Classify trend
if abs(percent_change) < 5:
    direction = "STABLE"
elif slope > 0:
    direction = "UPWARD"
else:
    direction = "DOWNWARD"

print("TREND ANALYSIS")
print("=" * 50)
print(f"Direction: {direction}")
print(f"Slope: {slope:.4f}")
print(f"Percent Change: {percent_change:.2f}%")
print(f"Start Value: {start_val:.2f}")
print(f"End Value: {end_val:.2f}")

## 6. Visualizing Trends with Trend Line

In [ ]:
# Plot with trend line
plt.figure(figsize=(12, 6))
plt.plot(df_sorted['date'], df_sorted['sales'], 
         linewidth=2, marker='o', markersize=4, 
         color='#2E86AB', label='Actual Sales')
plt.plot(df_sorted['date'], trend_line, 
         '--', linewidth=2, color='red', 
         alpha=0.7, label='Trend Line')

plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Sales', fontsize=12, fontweight='bold')
plt.title(f'Sales Over Time - {direction} Trend', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Detecting Anomalies and Changes

Identify unusual spikes, drops, or sudden changes.

In [ ]:
# Detect anomalies using z-score
mean = df_sorted['sales'].mean()
std = df_sorted['sales'].std()
threshold = 2.0

df_sorted['z_score'] = np.abs((df_sorted['sales'] - mean) / std)
anomalies = df_sorted[df_sorted['z_score'] > threshold]

print("ANOMALY DETECTION")
print("=" * 50)
print(f"Mean: {mean:.2f}")
print(f"Std Dev: {std:.2f}")
print(f"Threshold: {threshold} standard deviations")
print(f"\nAnomalies found: {len(anomalies)}")

if len(anomalies) > 0:
    print("\nAnomaly Details:")
    for idx, row in anomalies.iterrows():
        print(f"  {row['date'].date()}: {row['sales']:.2f} (z-score: {row['z_score']:.2f})")

## 8. Comprehensive Visualization with Anomalies

In [ ]:
# Create comprehensive plot
plt.figure(figsize=(14, 7))

# Plot main line
plt.plot(df_sorted['date'], df_sorted['sales'], 
         linewidth=2, marker='o', markersize=4, 
         color='#2E86AB', label='Sales', alpha=0.8)

# Plot trend line
plt.plot(df_sorted['date'], trend_line, 
         '--', linewidth=2, color='green', 
         alpha=0.7, label='Trend Line')

# Highlight anomalies
if len(anomalies) > 0:
    plt.scatter(anomalies['date'], anomalies['sales'], 
               color='red', s=150, marker='X', 
               label='Anomalies', zorder=5)

plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Sales', fontsize=12, fontweight='bold')
plt.title('Comprehensive Time-Series Analysis', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(loc='best')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("✓ Comprehensive visualization complete")

## 9. Multiple Time Series Comparison

In [ ]:
# Plot multiple series
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Sales plot
ax1.plot(df_sorted['date'], df_sorted['sales'], 
         linewidth=2, marker='o', markersize=3, color='#2E86AB')
ax1.set_ylabel('Sales', fontsize=12, fontweight='bold')
ax1.set_title('Sales Over Time', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Temperature plot
ax2.plot(df_sorted['date'], df_sorted['temperature'], 
         linewidth=2, marker='s', markersize=3, color='#A23B72')
ax2.set_xlabel('Date', fontsize=12, fontweight='bold')
ax2.set_ylabel('Temperature', fontsize=12, fontweight='bold')
ax2.set_title('Temperature Over Time', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("✓ Multiple time series visualized")

## 10. Key Takeaways

### What We Learned:
1. **Temporal Ordering**: Always sort data by time before analysis
2. **Line Plots**: Best for showing continuity and trends over time
3. **Trend Identification**: Classify as upward, downward, or stable
4. **Anomaly Detection**: Use statistical methods to find unusual patterns
5. **Visual Analysis**: Plots reveal patterns that numbers alone cannot show

### Why Line Plots for Time-Series?
- Show continuity between data points
- Emphasize temporal progression
- Make trends visually obvious
- Highlight sudden changes and anomalies
- Enable pattern recognition over time

### Common Pitfalls to Avoid:
- ❌ Treating time-based data as unordered
- ❌ Missing long-term trends due to lack of visualization
- ❌ Overreacting to short-term fluctuations
- ❌ Misinterpreting noise as meaningful patterns

## Practice Exercise

Try loading your own time-series data and:
1. Identify temporal columns
2. Create line plots
3. Identify trends
4. Detect anomalies
5. Interpret the patterns you observe

In [ ]:
# Your code here
